# N140 · 带权与可回滚并查集

**目标：** 在根关系之外维护相对势能并支持撤销。

**先修：** N083, N019, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 相对权值；路径压缩更新；回滚栈；按大小；回滚不直接套压缩；离线删边扩展。

## 从问题到算法

带权并查集中的weight[x]不是节点绝对值，而是x/parent[x]。路径压缩时必须同时乘上沿途比例，否则根虽找对，数值关系已损坏。回滚DSU解决另一种需求：恢复历史状态。为使修改数量可控，采用按大小合并而不做路径压缩，每次合并只记录常数个被改字段。

## 最小实现

**本章接口：** `WeightedDSU`、`RollbackDSU.union/find/snapshot/rollback`、`calc_equation(equations, values, queries)`

In [1]:
from math import isclose

class WeightedDSU:
    def __init__(self): self.parent={}; self.weight={}; self.size={}
    def add(self,x):
        if x not in self.parent: self.parent[x]=x; self.weight[x]=1.0; self.size[x]=1
    def find(self,x):
        if x not in self.parent: raise KeyError(x)
        path=[]; node=x
        while self.parent[node]!=node: path.append(node); node=self.parent[node]
        ratio=1.0
        for v in reversed(path): ratio*=self.weight[v]; self.weight[v]=ratio; self.parent[v]=node
        return node
    def union(self,x,y,value):
        if value==0: raise ValueError('nonzero ratio required')
        self.add(x); self.add(y); rx=self.find(x); ry=self.find(y); wx=self.weight[x]; wy=self.weight[y]
        if rx==ry:
            if not isclose(wx/wy,value,rel_tol=1e-9,abs_tol=1e-12): raise ValueError('inconsistent equation')
            return False
        if self.size[rx]<self.size[ry]:
            self.parent[rx]=ry; self.weight[rx]=value*wy/wx; self.size[ry]+=self.size[rx]
        else:
            self.parent[ry]=rx; self.weight[ry]=wx/(value*wy); self.size[rx]+=self.size[ry]
        return True
    def ratio(self,x,y):
        if x not in self.parent or y not in self.parent: return -1.0
        if self.find(x)!=self.find(y): return -1.0
        return self.weight[x]/self.weight[y]

def calc_equation(equations,values,queries):
    if len(equations)!=len(values): raise ValueError('one value per equation required')
    dsu=WeightedDSU()
    for (x,y),value in zip(equations,values): dsu.union(x,y,value)
    return [dsu.ratio(x,y) for x,y in queries]

class RollbackDSU:
    def __init__(self,n): self.parent=list(range(n)); self.size=[1]*n; self.components=n; self.history=[]
    def find(self,x):
        if not 0<=x<len(self.parent): raise IndexError('vertex outside DSU')
        while self.parent[x]!=x: x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b: self.history.append(None); return False
        if self.size[a]<self.size[b]: a,b=b,a
        self.history.append((a,b,self.size[a])); self.parent[b]=a; self.size[a]+=self.size[b]; self.components-=1
        return True
    def snapshot(self): return len(self.history)
    def rollback(self,snapshot):
        if not 0<=snapshot<=len(self.history): raise ValueError('snapshot outside active history')
        while len(self.history)>snapshot:
            change=self.history.pop()
            if change is None: continue
            a,b,old_size=change; self.parent[b]=b; self.size[a]=old_size; self.components+=1

## 正确性、前提与复杂度

设wx=x/rx，wy=y/ry，已知x/y=v，则rx/ry=v·wy/wx；反方向连接时取倒数。沿路径的比例相乘后仍等于节点对根的比值。回滚按相反顺序恢复父指针、旧size与分量数，正好逆转此前修改；无效union也记录占位，使快照对应操作边界。

**代价：** 带权DSU按大小加路径压缩，均摊近O(α(n))次结构操作；浮点误差按容差解释。回滚find/union O(log n)最坏，撤销t次union O(t)，空间O(n+历史长度)。快照只能回到当前历史祖先，回滚后被舍弃分支不能凭旧编号恢复。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

dsu=WeightedDSU(); dsu.union('a','b',2); dsu.union('b','c',3)
for x in dsu.parent: dsu.find(x)
show_table(['变量','根','变量/根'],[(x,dsu.parent[x],dsu.weight[x]) for x in dsu.parent])
print('a/c=',dsu.ratio('a','c'))
r=RollbackDSU(4); r.union(0,1); saved=r.snapshot(); r.union(1,2); print('回滚前分量',r.components); r.rollback(saved); print('回滚后分量',r.components)

变量,根,变量/根
a,a,1.0
b,a,0.5
c,a,0.16666666666666666


a/c= 6.0
回滚前分量 2
回滚后分量 3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert calc_equation([['a','b'],['b','c']],[2,3],[['a','c'],['c','a'],['x','x']])==[6.0,1/6,-1.0]
from random import Random
from collections import deque
rng=Random(140)
for _ in range(100):
    values=[rng.randrange(1,10) for _ in range(6)]; edges=[(rng.randrange(6),rng.randrange(6)) for _ in range(8)]; d=WeightedDSU(); adj={}
    for a,b in edges:
        ratio=values[a]/values[b]; d.union(a,b,ratio); adj.setdefault(a,[]).append((b,ratio)); adj.setdefault(b,[]).append((a,1/ratio))
    for a in range(7):
        for b in range(7):
            q=deque([(a,1.0)]) if a in adj else deque(); seen={a}; expected=-1.0
            while q:
                u,p=q.popleft()
                if u==b: expected=p; break
                for v,w in adj[u]:
                    if v not in seen: seen.add(v); q.append((v,p*w))
            assert isclose(d.ratio(a,b),expected,rel_tol=1e-9,abs_tol=1e-12)
r=RollbackDSU(8); edges=[]
for _ in range(300):
    if edges and rng.randrange(3)==0:
        target=rng.randrange(len(edges)+1); r.rollback(target); edges=edges[:target]
    else:
        a,b=rng.randrange(8),rng.randrange(8); r.union(a,b); edges.append((a,b))
    reach=[[i==j for j in range(8)] for i in range(8)]
    for a,b in edges: reach[a][b]=reach[b][a]=True
    for k in range(8):
        for i in range(8):
            for j in range(8): reach[i][j]|=reach[i][k] and reach[k][j]
    before=r.parent[:]
    for i in range(8):
        for j in range(8): assert (r.find(i)==r.find(j))==reach[i][j]
    assert r.parent==before and r.snapshot()==len(edges)
    assert r.components==len({r.find(i) for i in range(8)})
print('N140: 所有本章断言通过')

N140: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把比值改为差值约束。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 对同时间秘密传播比较BFS与临时合并，注明不同解法。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 399. Evaluate Division（canonical）
- 2092. Find All People With Secret（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。